In [0]:
# Step 1: Read Bronze Table

from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.retail.bronze_online_retail"
)

In [0]:
# Step 2: Initial Record Count

bronze_df.filter(
    F.col("Customer_ID").isNull()
).count()

243007

In [0]:
# Step 3: Remove Null Customer IDs

# Before:
silver_df = bronze_df.filter(
    F.col("Customer_ID").isNotNull()
)

In [0]:
#after:
silver_df.filter(
    F.col("Invoice").startswith("C")
).count()

18744

In [0]:
# Step 4: Remove Cancelled Invoices
# count before:
silver_df = silver_df.filter(
    ~F.col("Invoice").startswith("C")
)

In [0]:

# clean:
silver_df.filter(
    F.col("Quantity") <= 0
).count()

18744

In [0]:
# Step 5: Remove Negative Quantities
# Check:
silver_df = silver_df.filter(
    F.col("Quantity") > 0
)

In [0]:
# Clean:
silver_df.filter(
    F.col("Price") <= 0
).count()

71

In [0]:
# Step 6: Remove Invalid Prices

# Check:
silver_df = silver_df.filter(
    F.col("Price") > 0
)

In [0]:
# clean
silver_df = silver_df.dropDuplicates()

In [0]:
# Step 8: Standardize Invoice Date

# Check actual format first:

silver_df.select(
    "InvoiceDate"
).show(5, False)

+-------------------+
|InvoiceDate        |
+-------------------+
|2009-12-01 07:46:00|
|2009-12-01 09:08:00|
|2009-12-01 09:08:00|
|2009-12-01 09:24:00|
|2009-12-01 09:24:00|
+-------------------+
only showing top 5 rows


In [0]:
# Convert:
silver_df = silver_df.withColumn(
    "invoice_timestamp",
    F.to_timestamp(
        "InvoiceDate",
        "yyyy-MM-dd HH:mm:ss"
    )
)

In [0]:
# Step 9: Create Revenue Column

silver_df = silver_df.withColumn(
    "revenue",
    F.round(
        F.col("Quantity") * F.col("Price"),
        2
    )
)

In [0]:
# Step 10: Add Silver Metadata

silver_df = (
    silver_df
    .withColumn(
        "silver_load_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# Step 11: Validation

silver_df.select(
    F.min("revenue"),
    F.max("revenue")
).show()

+------------+------------+
|min(revenue)|max(revenue)|
+------------+------------+
|         0.0|    168469.6|
+------------+------------+



In [0]:
# Data Quality Summary

print(
    f"Silver Record Count: {silver_df.count()}"
)

Silver Record Count: 779425


In [0]:
# Step 12: Write Silver Layer

(
    silver_df.write
             .format("delta")
             .mode("overwrite")
             .saveAsTable(
                 "workspace.retail.silver_online_retail"
             )
)

In [0]:
# Step 13: Validate

silver_table = spark.table(
    "workspace.retail.silver_online_retail"
)

silver_table.count()

779425


- Bronze 1,067,371 records
-        ↓

- Remove Null Customers
- Remove Cancelled Orders
- Remove Invalid Prices
- Remove Negative Quantities
- Remove Duplicates
- Create Revenue
- Standardize Dates

       ↓

Silver
~780k-900k clean records